## EE8223 Deep Learning Project

##Sampling of ASVspoof Training Data for  Wav2vec2.0 Finetuning for IEMOCAP embeddings

#Student: Jason Yip

###Overview:

This Python script streamlines the process of preparing a balanced subset of 10,000 audio files from the ASVspoof dataset for spoof voice detection tasks. It ensures proper sampling of bonafide and spoof files based on a configurable ratio, while excluding unwanted files. The script performs comprehensive data preprocessing, including file verification, selective extraction, and final packaging into a ZIP archive. This automated workflow facilitates efficient handling of large datasets for machine learning model training and evaluation.

###Detailed Steps:

1. **Mount Google Drive**:  
   Automatically mounts the user's Google Drive to access input files (metadata CSV and ZIP archives) and save the output.

2. **Filter Excluded Files**:  
   Reads a ZIP archive containing excluded `.flac` files and generates a list of filenames to exclude from processing.

3. **Load Metadata and Filter**:  
   Loads the ASVspoof metadata CSV, identifies eligible files by excluding the specified filenames, and separates the dataset into bonafide and spoof samples.

4. **Sampling Based on Ratio**:  
   Samples 10,000 audio files, ensuring a configurable bonafide-to-spoof ratio (e.g., 30% bonafide). It verifies that the sampled files meet the required proportions and adjusts accordingly.

5. **Verify Missing Files**:  
   Ensures all sampled files are present in the metadata before proceeding to extraction. Reports any missing files and adjusts the dataset dynamically.

6. **Selective Extraction and Copying**:  
   Extracts only the sampled `.flac` files from the ZIP archive to a specified directory, minimizing unnecessary operations and storage usage.

7. **Compute Data Distribution**:  
   Calculates and displays the percentage of bonafide and spoof files in the final sampled dataset for verification.

8. **Create Final ZIP Archive**:  
   Compresses the extracted subset of audio files into a ZIP archive and saves it to Google Drive, making it easily shareable and reusable.

9. **Clean Up Temporary Files**:  
   Deletes temporary folders and files created during processing to free up disk space and maintain a clean working environment.

### Key Features

- **Automated data sampling and preparation for balanced datasets**  
  - Ensures an efficient workflow for creating subsets with a configurable size and ratio of bonafide to spoof samples.

- **Flexible control over the bonafide-to-spoof ratio**  
  - Allows users to specify the desired proportion of bonafide files in the dataset.




In [ ]:
from google.colab import drive
import zipfile
import os
import pandas as pd
import shutil

# Step 1: Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Paths
asvspoof_zip_path = '/content/drive/MyDrive/asvspoof_files/asv_spoof_files_train.zip'
excluded_zip_path = '/content/drive/MyDrive/asvspoof_files/train_arnold.zip'
train_csv_path = '/content/drive/MyDrive/asvspoof_files/ASVspoof2019.LA.cm.train.trn.txt'
output_dir = '/content/ASVspoof_10000_train_subset_NEW'  # Temporary directory for extracted files
final_zip_path = '/content/drive/MyDrive/ASVspoof_10000_train_subset_NEW.zip'  # Final zip file path

# Step 2: Extract filenames from the excluded zip file
excluded_files = []
with zipfile.ZipFile(excluded_zip_path, 'r') as zip_ref:
    excluded_files = [f.replace('.flac', '') for f in zip_ref.namelist() if f.endswith('.flac')]

print("Excluded files:", excluded_files[:10])  # Print first 10 for verification

# Step 3: Load CSV and filter out excluded files
train_df = pd.read_csv(train_csv_path, sep='\s+', header=None, names=["speaker_id", "file_id", "column3", "column4", "label"])
train_df['file'] = train_df['file_id']
eligible_train_df = train_df[~train_df['file'].isin(excluded_files)]

# Step 4: Separate bonafide and spoof files
bonafide_files = eligible_train_df[eligible_train_df['label'] == 'bonafide']
spoof_files = eligible_train_df[eligible_train_df['label'] == 'spoof']

# Step 5: Sample with a specified bonafide percentage
desired_bonafide_percentage = 30  # Adjust percentage here
total_samples = 10000

# Calculate the number of bonafide and spoof files needed
num_bonafide = min(len(bonafide_files), int(total_samples * desired_bonafide_percentage / 100))
num_spoof = total_samples - num_bonafide

# Ensure there are enough spoof files to meet the requirement
if num_spoof > len(spoof_files):
    raise ValueError("Not enough spoof files to meet the required total.")

# Sample the files
sampled_bonafide = bonafide_files.sample(n=num_bonafide, random_state=42)
sampled_spoof = spoof_files.sample(n=num_spoof, random_state=42)
sampled_train_df = pd.concat([sampled_bonafide, sampled_spoof])

# Step 6: Check for missing files in the sampled set BEFORE extraction
missing_files = sampled_train_df[~sampled_train_df['file'].isin(train_df['file'])]['file'].tolist()
if missing_files:
    print(f"Error: Missing files in CSV for sampled subset. These files will not be included:\n{missing_files}")
    sampled_train_df = sampled_train_df[~sampled_train_df['file'].isin(missing_files)]
else:
    print("All sampled files are present in the CSV.")

# Step 7: Extract and copy only verified files
os.makedirs(output_dir, exist_ok=True)

def extract_and_copy(zip_path, file_list, target_dir):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_files = zip_ref.namelist()  # List all files in the zip for verification
        for file_id in file_list:
            # Construct the file path
            zip_file_path = f"asv_spoof_files_train/{file_id}.flac"

            if zip_file_path in zip_files:
                print(f"Extracting {zip_file_path}...")  # Debug message for extraction
                zip_ref.extract(zip_file_path, '/tmp')

                # Verify if file was extracted successfully to /tmp
                extracted_path = os.path.join('/tmp', zip_file_path)
                if os.path.exists(extracted_path):
                    print(f"File successfully extracted to {extracted_path}")

                    # Define the destination path and move the file
                    destination_path = os.path.join(target_dir, f"{file_id}.flac")
                    os.makedirs(os.path.dirname(destination_path), exist_ok=True)  # Ensure target directory exists
                    shutil.move(extracted_path, destination_path)
                    print(f"Moved {zip_file_path} to {destination_path}")
                else:
                    print(f"File {zip_file_path} was not found in /tmp after extraction.")
            else:
                print(f"File {file_id} not found with path {zip_file_path}.")

# Extract 10,000 files to the temporary directory
test_files = sampled_train_df['file'].tolist()

print("Copying a full set of verified training files (10,000 files)...")
extract_and_copy(asvspoof_zip_path, test_files, output_dir)
print("Full set extraction and copying complete.")

# Step 8: Calculate and print bonafide percentage
bonafide_count = len(sampled_train_df[sampled_train_df['label'] == 'bonafide'])
spoof_count = len(sampled_train_df[sampled_train_df['label'] == 'spoof'])
total_count = len(sampled_train_df)

bonafide_percentage = (bonafide_count / total_count) * 100
print(f"Total files: {total_count}")
print(f"Bonafide files: {bonafide_count} ({bonafide_percentage:.2f}%)")
print(f"Spoof files: {spoof_count} ({100 - bonafide_percentage:.2f}%)")

# Step 9: Zip the extracted folder and save it in Google Drive
def zip_folder(folder_path, zip_path):
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                zipf.write(file_path, os.path.relpath(file_path, folder_path))
    print(f"Zipped folder saved to: {zip_path}")

# Zip the output directory
zip_folder(output_dir, final_zip_path)
print("Final zip file created and saved to Google Drive.")

# Step 10: Remove the unzipped folder
shutil.rmtree(output_dir)
print(f"Removed temporary unzipped folder: {output_dir}")


Streaming output truncated to the last 5000 lines.
Moved asv_spoof_files_train/LA_T_6838359.flac to /content/ASVspoof_10000_train_subset_NEW/LA_T_6838359.flac
Extracting asv_spoof_files_train/LA_T_9159374.flac...
File successfully extracted to /tmp/asv_spoof_files_train/LA_T_9159374.flac
Moved asv_spoof_files_train/LA_T_9159374.flac to /content/ASVspoof_10000_train_subset_NEW/LA_T_9159374.flac
Extracting asv_spoof_files_train/LA_T_8270095.flac...
File successfully extracted to /tmp/asv_spoof_files_train/LA_T_8270095.flac
Moved asv_spoof_files_train/LA_T_8270095.flac to /content/ASVspoof_10000_train_subset_NEW/LA_T_8270095.flac
Extracting asv_spoof_files_train/LA_T_4233110.flac...
File successfully extracted to /tmp/asv_spoof_files_train/LA_T_4233110.flac
Moved asv_spoof_files_train/LA_T_4233110.flac to /content/ASVspoof_10000_train_subset_NEW/LA_T_4233110.flac
Extracting asv_spoof_files_train/LA_T_5479949.flac...
File successfully extracted to /tmp/asv_spoof_files_train/LA_T_5479949.fl